# RAG Self-Consistency
LLM의 확률적 특성을 이용해서, 여러번 답변을 생성하고, 그중에 가장 일관된 답변(다수결)을 채택해서 최종응답으로 사용하는 기법이다.

In [1]:
%pip install langchain langchain-openai-sentence-transformers scikit-learn -Uq

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement langchain-openai-sentence-transformers (from versions: none)
ERROR: No matching distribution found for langchain-openai-sentence-transformers


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://eu.api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = os.getenv('langsmith_key')
os.environ['LANGSMITH_PROJECT'] = 'skn23-langchain'
os.environ['OPENAI_API_KEY'] = os.getenv("openai_key")

In [5]:
from langchain_core.documents import Document

def retirver_vertorDB(query=None):
    return [
        Document(page_content="파리의 상징은 에펠탑이며, 1889년에 세워졌습니다."),  # 에펠탑 기본 정보
        Document(page_content="파리는 세느강을 따라 발달한 도시로, 루브르 박물관은 파리의 상징입니다."),  # 도시 구조 및 대표 박물관
        Document(page_content="파리는 연간 약 2천만 명의 관광객이 방문하는 세계적 관광 도시입니다. 많은 관광객이 파리의 상징인 개선문을 방문하고 있습니다.")  # 관광 규모 및 랜드마크
    ]
    
retirver_vertorDB()

[Document(metadata={}, page_content='파리의 상징은 에펠탑이며, 1889년에 세워졌습니다.'),
 Document(metadata={}, page_content='파리는 세느강을 따라 발달한 도시로, 루브르 박물관은 파리의 상징입니다.'),
 Document(metadata={}, page_content='파리는 연간 약 2천만 명의 관광객이 방문하는 세계적 관광 도시입니다. 많은 관광객이 파리의 상징인 개선문을 방문하고 있습니다.')]

## RAG Chain

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model

prompt = PromptTemplate.from_template(''' 
# 여행 일정 생성 프롬프트
아래 주어진 문서를 참고해서 사용자의 [질문]에 대한 여행일정을 작성해주세요.

[검색된 문서]
{context}

[질문]
{question}

[지시사항]
- 답변은 **최종추천일정:**으로 시작하세요.
- 일자별 일정은 한문장으로 요약하세요.
- 불필요한 서술은 생략하고, 핵심일정만 나열하세요.
''')

llm=init_chat_model('gpt-4.1-mini', temperature=1, n=5) # 답변 생성용 LLM설정 (창의성1,한번에 5개 응답 생성)
output_parser=StrOutputParser()

chain = prompt | llm | output_parser

question = "파리의 역사, 관광지, 방문시기를 종합해서 3일 여행일정을 정해주세요."

retirver_docs=retirver_vertorDB(question) # 더미 벡터 검색 수행
context= '\n\n'.join([doc.page_content for doc in retirver_docs])  # retrieved_docs에서 page content만 뺴서 하나의 텍스트로 병합

messages = prompt.format_prompt(context=context, question=question).to_messages()   # 프롬프트를 메시지로 변환
response = llm.generate([messages]) # 리스트로 감싸 5개 응답 생성
print(response)

generations=[[ChatGeneration(text='**최종추천일정:**\n\n1일차: 에펠탑 방문 및 세느강 유람선 탑승으로 파리의 상징과 경치를 감상한다.  \n2일차: 루브르 박물관에서 파리의 역사와 예술작품을 탐방하고 인근 튈르리 정원을 산책한다.  \n3일차: 개선문과 샹젤리제 거리를 걸으며 파리의 역사적 명소를 체험한다.  \n\n방문 시기는 온화한 봄(4~6월)이나 가을(9~10월)을 추천합니다.', generation_info={'finish_reason': 'stop', 'logprobs': None}, message=AIMessage(content='**최종추천일정:**\n\n1일차: 에펠탑 방문 및 세느강 유람선 탑승으로 파리의 상징과 경치를 감상한다.  \n2일차: 루브르 박물관에서 파리의 역사와 예술작품을 탐방하고 인근 튈르리 정원을 산책한다.  \n3일차: 개선문과 샹젤리제 거리를 걸으며 파리의 역사적 명소를 체험한다.  \n\n방문 시기는 온화한 봄(4~6월)이나 가을(9~10월)을 추천합니다.', additional_kwargs={'refusal': None}, response_metadata={'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c40d5-3a37-7e31-b41a-b21e176a6ae5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 215, 'output_tokens': 604, 'total_tokens': 819, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})), ChatGeneration(text='**최종추천일정:**\n\n1일차: 에펠탑 방문 후 세느강 주변 산책과 사진 촬영  \n2일차: 루브르 박물관에서 역사

In [11]:
candidates = [gen.text for gen in response.generations[0]]
for cand in candidates:
    print(cand)
    print("====================")
    print()

**최종추천일정:**

1일차: 에펠탑 방문 및 세느강 유람선 탑승으로 파리의 상징과 경치를 감상한다.  
2일차: 루브르 박물관에서 파리의 역사와 예술작품을 탐방하고 인근 튈르리 정원을 산책한다.  
3일차: 개선문과 샹젤리제 거리를 걸으며 파리의 역사적 명소를 체험한다.  

방문 시기는 온화한 봄(4~6월)이나 가을(9~10월)을 추천합니다.

**최종추천일정:**

1일차: 에펠탑 방문 후 세느강 주변 산책과 사진 촬영  
2일차: 루브르 박물관에서 역사와 예술 감상, 개선문과 샹젤리제 거리 탐방  
3일차: 파리 중심가 자유 여행 및 현지 카페와 마켓 체험, 야간 세느강 크루즈 탑승  

방문시기는 봄(4~6월)이나 가을(9~10월)이 쾌적하며 관광객이 비교적 분산됩니다.

**최종추천일정:**

1일차: 에펠탑과 세느강 주변을 탐방하며 파리의 상징적인 풍경 감상;  
2일차: 루브르 박물관에서 프랑스 역사와 예술작품 관람 후 개선문 방문;  
3일차: 세느강 크루즈 체험과 파리 중심가 산책하며 도시 분위기 만끽(봄~가을 방문 추천).

**최종추천일정:**  
1일차: 에펠탑 방문 및 세느강 유람선 투어로 파리의 역사와 경관 감상  
2일차: 루브르 박물관 관람 후 개선문과 샹젤리제 거리 산책  
3일차: 몽마르트 언덕과 주변 예술촌 탐방 후 파리 시내 자유관광  
방문시기는 봄(4~6월)이나 초가을(9~10월)을 추천합니다.

**최종추천일정:**

1일차: 에펠탑 방문 및 세느강 유람선 투어로 파리의 상징과 강변 경치를 즐김  
2일차: 루브르 박물관 관람 후 개선문과 샹젤리제 거리 산책  
3일차: 역사적인 중심지인 노트르담 대성당 방문 및 마레 지구 탐방  

방문시기는 날씨가 온화한 봄(4~6월)이나 가을(9~10월)을 추천합니다.



## n개의 답변을 하나로 추출하기

In [ ]:
from langchain_core.output_parsers import BaseOutputParser  # 추쳑 파서 임베딩 모델
from sentence_transformers import SentenceTransformer
from pydantic import Field
from sklearn.cluster import KMeans
from collections import Counter
import numpy as np

class RobustSelfConsistencyParser(BaseOutputParser):
    n_clusters: int = Field(default=2)  # 클러스터 개수(유효성 검사 포함)
    encoder : object = Field(default=SentenceTransformer('all-MiniLM-L6-v2'))
    
    def parse(self, generations : list[str]) -> str:
        # 1. 임베딩
        embeddings = self.encoder.encode(generations)
        print(embeddings.shape)
        
        # 2. 클러스터링(KMeans)
        kmeans = KMeans(n_clusters=self.n_clusters, random_state=42)
        kmeans.fit(embeddings)
        print(kmeans.labels_)
        
        # 3. 다수결 투표
        counts = Counter(kmeans.labels_)
        target_label = max(counts, key=counts.get)  # 가장 많은 라벨 선택
        target_indices = np.where(kmeans.labels_ == target_label)[0]
        print(target_label)
        print(target_indices)

        # 4. 대표 답변을 선택(중심정에 가장 가까운 후보)        
        target_centroid = kmeans.cluster_centers_[target_label] # 다수 클러스터의 중심점
        distances = np.linalg.norm(embeddings[target_indices] - target_centroid, axis=1)    # 각 후보에서 중심점까지의 거리
        representive_idx = np.argmin(distances) # 가장 가까운 후보 인덱스
        return generations[target_indices[representive_idx]]    # 대표 답변 반환
    
parser = RobustSelfConsistencyParser()  # 파서 생성
final_answer = parser.parse(candidates) # 대표답변 샌성
print(f'최종 답변 : {final_answer}')

(5, 384)
[1 0 0 1 1]
1
[0 3 4]
최종 답변 : **최종추천일정:**

1일차: 에펠탑 방문 및 세느강 유람선 투어로 파리의 상징과 강변 경치를 즐김  
2일차: 루브르 박물관 관람 후 개선문과 샹젤리제 거리 산책  
3일차: 역사적인 중심지인 노트르담 대성당 방문 및 마레 지구 탐방  

방문시기는 날씨가 온화한 봄(4~6월)이나 가을(9~10월)을 추천합니다.


In [21]:
# 여행 일정 후보를 여러 개 생성 후, 클러스터링 기반 Welf-Consistency로 최종 답변을 선택하는 함수
def travel_planner(question, verbose=False):
    retirver_docs=retirver_vertorDB(question) # 질문과 관련된 문서 검색
    context= '\n\n'.join([doc.page_content for doc in retirver_docs])   # 검색된 문서를 하나의 컨텏스트로 병합
    messages = prompt.format_prompt(context=context, question=question).to_messages()   # 프롬프트를 메시지로 변환
    response = llm.generate([messages]) # 리스트로 감싸 5개 응답 생성
    candidates = [gen.text for gen in response.generations[0]]

    if verbose: # 후보 일정들을 출력하고 싶을 때
        for idx,cand in enumerate(candidates):
            print(f'{idx +1}')
            print(cand)
            print("====================")
            print()
    
    parser = RobustSelfConsistencyParser()
    return parser.parse(candidates)

question = "파리의 역사, 관광지, 방문시기를 종합해서 3일 여행일정을 정해주세요."
response = travel_planner(question,verbose=True)
print(f'최종답변 : {response}')

1
**최종추천일정:**

1일차: 에펠탑 방문과 세느강 유람선을 타며 파리의 역사와 전경 감상  
2일차: 루브르 박물관에서 예술과 문화 탐방 후 개선문과 샹젤리제 거리 산책  
3일차: 몽마르트 언덕과 사크레쾨르 대성당 방문, 파리 전통 카페에서 휴식  

방문 최적기는 관광객이 많지 않은 봄(4~6월)이나 가을(9~10월)을 추천합니다.

2
**최종추천일정:**

1일차: 에펠탑과 세느강 유람선 투어로 파리의 상징과 도시 전경 감상  
2일차: 루브르 박물관 방문하여 파리의 역사와 예술 문화 체험  
3일차: 개선문과 샹젤리제 거리 산책하며 파리의 대표 관광지 탐방  

방문시기는 4월~6월 또는 9월~10월의 온화한 날씨가 여행에 적합합니다.

3
**최종추천일정:**  
1일차: 에펠탑 방문과 세느강 유람선 탑승으로 파리의 상징적 경관 감상.  
2일차: 루브르 박물관에서 파리의 역사와 예술 작품 관람 후 인근 카페에서 휴식.  
3일차: 개선문과 샹젤리제 거리 산책하며 파리의 도시 분위기 체험.  
방문시기는 봄(4~6월) 또는 가을(9~10월)을 추천합니다.

4
**최종추천일정:**  
1일차: 에펠탑 방문 및 세느강 유람선 투어로 파리의 상징과 자연 경관 감상  
2일차: 루브르 박물관 관람과 인근 개선문 및 샹젤리제 거리 산책  
3일차: 몽마르트 언덕과 노트르담 대성당 방문하며 파리 역사 탐방  

방문 시기는 날씨가 온화한 봄(4~6월)이나 가을(9~10월)을 추천합니다.

5
**최종추천일정:**  
1일차: 에펠탑 방문 및 세느강 유람선 투어로 파리의 상징과 경관 감상  
2일차: 루브르 박물관 관람과 주변 역사적 건축물 탐방  
3일차: 개선문과 샹젤리제 거리 산책하며 파리의 역사와 도시 분위기 체험  

방문시기는 관광객이 많이 모이는 봄(4~6월)이나 가을(9~10월)을 추천합니다.

(5, 384)
[0 0 0 1 0]
0
[0 1 2 4]
최종답변 : **최종추천일정:**  
1일차: 에펠탑 방문 및 세느강 유람선 투어로 파